In [1]:
import spacy
from spacy import displacy
from spacy.tokens import Doc, Span, Token

from biz.dfch.asdste100vocab import Vocab
from biz.dfch.asdste100vocab import Word
from biz.dfch.asdste100vocab import WordCategory
from biz.dfch.asdste100vocab import WordStatus
from biz.dfch.asdste100vocab import WordType

from biz.dfch.ste100parser import GrammarType
from biz.dfch.ste100parser import Inspector
from biz.dfch.ste100parser import Parser
from biz.dfch.ste100parser import ParserAction
from biz.dfch.ste100parser import Ste100Doc
from biz.dfch.ste100parser.serializer.text_interpreter import TextInterpreter

from biz.dfch.ste100parser.serializer.token_base import TokenBase
from biz.dfch.ste100parser.serializer.token_base import ListToken

from biz.dfch.ste100parser.token_registry import TokenRegistry
from biz.dfch.ste100parser.rule_registry import RuleRegistry
from biz.dfch.ste100parser.rule_registry import RuleContext
from biz.dfch.ste100parser.rule_registry import TextUtils

nlp = spacy.load("en_core_web_sm")


In [2]:
text = """The button was pressed 2 times. The end user presses the button two times. The pressed button is orange."""
text = """During transmission, the data was corrupted.
During transmission, something corrupted the data.
Transmission corrupted the data.
The safety procedures are given by the manufacturer.
The main gear leg is held by the side stay.
  * The volume control can be adjusted.

A) The valve will be adjusted during the test.
  * The oil temperature must be adjusted before the start of the test.

"""

parser = Parser(GrammarType.ASD_STE100_9)
tree = parser.invoke(text, action=ParserAction.PASS2)
vocab = Vocab()

interpreter = TextInterpreter(vocab=vocab)
tokens = interpreter.invoke(tree)
doc = Ste100Doc(tokens)
inspector = Inspector()
structure = inspector.ste100doc(doc)
print(structure)

token_registry = TokenRegistry.Factory.get_instance()
rule_context = RuleContext(vocab, token_registry, TextUtils())
rule_registry = RuleRegistry()
rule_registry.install_rules("biz.dfch.ste100parser.rule_repository")

def _process_tokens(tokens: list[TokenBase]):
    for token in tokens:
        if isinstance(token, ListToken):
            _process_tokens(token.tokens)
        # print(f"[{type(token).__name__}] Processing token '{token.text}' ...")
        rules = rule_registry.get_rules(token)
        for rule in rules:
            # print(f"Processing rule '{rule.rule_id}' [{rule.priority}] [{type(token).__name__}] ...")
            try:
                test_results = rule.examine(token, rule_context)
                for test_result in test_results:
                    print(f"[{test_result.severity}] {test_result.rule_id}: '{test_result.message}'")
            except Exception as ex:
                print(f"'{type(rule).__name__}' failed [{ex}]")

_process_tokens(list(doc))

# displacy.render(doc, jupyter=True)


#6 [TEXT]: 'During'.
#1 [WS]: ' '.
#13 [TEXT]: 'transmission,'.
#1 [WS]: ' '.
#3 [TEXT]: 'the'.
#1 [WS]: ' '.
#4 [TEXT]: 'data'.
#1 [WS]: ' '.
#3 [TEXT]: 'was'.
#1 [WS]: ' '.
#10 [TEXT]: 'corrupted.'.
#1 [LINEBREAK]: '
'.
#6 [TEXT]: 'During'.
#1 [WS]: ' '.
#13 [TEXT]: 'transmission,'.
#1 [WS]: ' '.
#9 [TEXT]: 'something'.
#1 [WS]: ' '.
#9 [TEXT]: 'corrupted'.
#1 [WS]: ' '.
#3 [TEXT]: 'the'.
#1 [WS]: ' '.
#5 [TEXT]: 'data.'.
#1 [LINEBREAK]: '
'.
#12 [TEXT]: 'Transmission'.
#1 [WS]: ' '.
#9 [TEXT]: 'corrupted'.
#1 [WS]: ' '.
#3 [TEXT]: 'the'.
#1 [WS]: ' '.
#5 [TEXT]: 'data.'.
#1 [LINEBREAK]: '
'.
#3 [TEXT]: 'The'.
#1 [WS]: ' '.
#6 [TEXT]: 'safety'.
#1 [WS]: ' '.
#10 [TEXT]: 'procedures'.
#1 [WS]: ' '.
#3 [TEXT]: 'are'.
#1 [WS]: ' '.
#5 [TEXT]: 'given'.
#1 [WS]: ' '.
#2 [TEXT]: 'by'.
#1 [WS]: ' '.
#3 [TEXT]: 'the'.
#1 [WS]: ' '.
#13 [TEXT]: 'manufacturer.'.
#1 [LINEBREAK]: '
'.
#3 [TEXT]: 'The'.
#1 [WS]: ' '.
#4 [TEXT]: 'main'.
#1 [WS]: ' '.
#4 [TEXT]: 'gear'.
#1 [WS]: ' '.
#3 [TEXT]: 'le

[WARNING] R3.6: 'Descriptive: [['data', 'was', 'corrupted']] Use the active voice. In descriptive writing, you can use the passive voice only when the agent is unknown.'
[0/13] 'During' [Word]
[8] spacy: '['During', 'transmission', ',', 'something', 'corrupted', 'the', 'data', '.']'
[2/13] 'transmission' [Text]
[8] spacy: '['During', 'transmission', ',', 'something', 'corrupted', 'the', 'data', '.']'
[5/13] 'something' [Word]
[8] spacy: '['During', 'transmission', ',', 'something', 'corrupted', 'the', 'data', '.']'
[7/13] 'corrupted' [Text]
[8] spacy: '['During', 'transmission', ',', 'something', 'corrupted', 'the', 'data', '.']'
[9/13] 'the' [Word]
[8] spacy: '['During', 'transmission', ',', 'something', 'corrupted', 'the', 'data', '.']'
[11/13] 'data' [Word]
[8] spacy: '['During', 'transmission', ',', 'something', 'corrupted', 'the', 'data', '.']'
'During' [ADP] [prep]
'transmission' [NOUN] [pobj]
',' [PUNCT] [punct]
'something' [PRON] [nsubj]
'corrupted' [VERB] [ROOT]
'the' [DET] [d

[0/8] 'Transmission' [Text]
[5] spacy: '['Transmission', 'corrupted', 'the', 'data', '.']'
[2/8] 'corrupted' [Text]
[5] spacy: '['Transmission', 'corrupted', 'the', 'data', '.']'
[4/8] 'the' [Word]
[5] spacy: '['Transmission', 'corrupted', 'the', 'data', '.']'
[6/8] 'data' [Word]
[5] spacy: '['Transmission', 'corrupted', 'the', 'data', '.']'
'Transmission' [NOUN] [nsubj]
'corrupted' [VERB] [ROOT]
'the' [DET] [det]
'data' [NOUN] [dobj]
'.' [PUNCT] [punct]


[0/16] 'The' [Word]
[9] spacy: '['The', 'safety', 'procedures', 'are', 'given', 'by', 'the', 'manufacturer', '.']'
[2/16] 'safety' [Word]
[9] spacy: '['The', 'safety', 'procedures', 'are', 'given', 'by', 'the', 'manufacturer', '.']'
[4/16] 'procedures' [Text]
[9] spacy: '['The', 'safety', 'procedures', 'are', 'given', 'by', 'the', 'manufacturer', '.']'
[6/16] 'are' [Text]
[9] spacy: '['The', 'safety', 'procedures', 'are', 'given', 'by', 'the', 'manufacturer', '.']'
[8/16] 'given' [Text]
[9] spacy: '['The', 'safety', 'procedures', 'are', 'given', 'by', 'the', 'manufacturer', '.']'
[10/16] 'by' [Word]
[9] spacy: '['The', 'safety', 'procedures', 'are', 'given', 'by', 'the', 'manufacturer', '.']'
[12/16] 'the' [Word]
[9] spacy: '['The', 'safety', 'procedures', 'are', 'given', 'by', 'the', 'manufacturer', '.']'
[14/16] 'manufacturer' [Text]
[9] spacy: '['The', 'safety', 'procedures', 'are', 'given', 'by', 'the', 'manufacturer', '.']'
'The' [DET] [det]
'safety' [NOUN] [compound]
'procedures'

[WARNING] R3.6: 'Descriptive: [['procedures', 'are', 'given']] Use the active voice. In descriptive writing, you can use the passive voice only when the agent is unknown.'
[0/20] 'The' [Word]
[11] spacy: '['The', 'main', 'gear', 'leg', 'is', 'held', 'by', 'the', 'side', 'stay', '.']'
[2/20] 'main' [Word]
[11] spacy: '['The', 'main', 'gear', 'leg', 'is', 'held', 'by', 'the', 'side', 'stay', '.']'
[4/20] 'gear' [Text]
[11] spacy: '['The', 'main', 'gear', 'leg', 'is', 'held', 'by', 'the', 'side', 'stay', '.']'
[6/20] 'leg' [Text]
[11] spacy: '['The', 'main', 'gear', 'leg', 'is', 'held', 'by', 'the', 'side', 'stay', '.']'
[8/20] 'is' [Text]
[11] spacy: '['The', 'main', 'gear', 'leg', 'is', 'held', 'by', 'the', 'side', 'stay', '.']'
[10/20] 'held' [Text]
[11] spacy: '['The', 'main', 'gear', 'leg', 'is', 'held', 'by', 'the', 'side', 'stay', '.']'
[12/20] 'by' [Word]
[11] spacy: '['The', 'main', 'gear', 'leg', 'is', 'held', 'by', 'the', 'side', 'stay', '.']'
[14/20] 'the' [Word]
[11] spacy: '

[WARNING] R3.6: 'Descriptive: [['leg', 'is', 'held']] Use the active voice. In descriptive writing, you can use the passive voice only when the agent is unknown.'
[0/12] 'The' [Word]
[7] spacy: '['The', 'volume', 'control', 'can', 'be', 'adjusted', '.']'
[2/12] 'volume' [Word]
[7] spacy: '['The', 'volume', 'control', 'can', 'be', 'adjusted', '.']'
[4/12] 'control' [Word]
[7] spacy: '['The', 'volume', 'control', 'can', 'be', 'adjusted', '.']'
[6/12] 'can' [Word]
[7] spacy: '['The', 'volume', 'control', 'can', 'be', 'adjusted', '.']'
[8/12] 'be' [Word]
[7] spacy: '['The', 'volume', 'control', 'can', 'be', 'adjusted', '.']'
[10/12] 'adjusted' [Text]
[7] spacy: '['The', 'volume', 'control', 'can', 'be', 'adjusted', '.']'
'The' [DET] [det]
'volume' [NOUN] [compound]
'control' [NOUN] [nsubjpass]
'can' [AUX] [aux]
'be' [AUX] [auxpass]
'adjusted' [VERB] [ROOT]
'.' [PUNCT] [punct]


[WARNING] R3.6: 'Descriptive: [['control', 'be', 'adjusted']] Use the active voice. In descriptive writing, you can use the passive voice only when the agent is unknown.'
[0/16] 'The' [Word]
[9] spacy: '['The', 'valve', 'will', 'be', 'adjusted', 'during', 'the', 'test', '.']'
[2/16] 'valve' [Text]
[9] spacy: '['The', 'valve', 'will', 'be', 'adjusted', 'during', 'the', 'test', '.']'
[4/16] 'will' [Word]
[9] spacy: '['The', 'valve', 'will', 'be', 'adjusted', 'during', 'the', 'test', '.']'
[6/16] 'be' [Word]
[9] spacy: '['The', 'valve', 'will', 'be', 'adjusted', 'during', 'the', 'test', '.']'
[8/16] 'adjusted' [Text]
[9] spacy: '['The', 'valve', 'will', 'be', 'adjusted', 'during', 'the', 'test', '.']'
[10/16] 'during' [Word]
[9] spacy: '['The', 'valve', 'will', 'be', 'adjusted', 'during', 'the', 'test', '.']'
[12/16] 'the' [Word]
[9] spacy: '['The', 'valve', 'will', 'be', 'adjusted', 'during', 'the', 'test', '.']'
[14/16] 'test' [Word]
[9] spacy: '['The', 'valve', 'will', 'be', 'adjusted'

[ERROR] R3.6: 'Procedural: [['valve', 'be', 'adjusted']] Use the active voice. In descriptive writing, you can use the passive voice only when the agent is unknown.'
[0/24] 'The' [Word]
[13] spacy: '['The', 'oil', 'temperature', 'must', 'be', 'adjusted', 'before', 'the', 'start', 'of', 'the', 'test', '.']'
[2/24] 'oil' [Word]
[13] spacy: '['The', 'oil', 'temperature', 'must', 'be', 'adjusted', 'before', 'the', 'start', 'of', 'the', 'test', '.']'
[4/24] 'temperature' [Text]
[13] spacy: '['The', 'oil', 'temperature', 'must', 'be', 'adjusted', 'before', 'the', 'start', 'of', 'the', 'test', '.']'
[6/24] 'must' [Word]
[13] spacy: '['The', 'oil', 'temperature', 'must', 'be', 'adjusted', 'before', 'the', 'start', 'of', 'the', 'test', '.']'
[8/24] 'be' [Word]
[13] spacy: '['The', 'oil', 'temperature', 'must', 'be', 'adjusted', 'before', 'the', 'start', 'of', 'the', 'test', '.']'
[10/24] 'adjusted' [Text]
[13] spacy: '['The', 'oil', 'temperature', 'must', 'be', 'adjusted', 'before', 'the', 'sta

[ERROR] R3.6: 'Procedural: [['temperature', 'be', 'adjusted']] Use the active voice. In descriptive writing, you can use the passive voice only when the agent is unknown.'


In [ ]:
text = """Close the door.
You close the door.
The door is closed.
Turn off the engine.
Do not enter.

A) Close the door.
B) You close the door.
C) The door is closed.
D) Turn off the engine.
E) Do not enter.

"""

text = """Do this "test" at full (100 units) speed."""
parser = Parser(GrammarType.ASD_STE100_9)
tree = parser.invoke(text, action=ParserAction.PASS2)
vocab = Vocab()

interpreter = TextInterpreter(vocab=vocab)
tokens = interpreter.invoke(tree)
doc = Ste100Doc(tokens)
inspector = Inspector()
structure = inspector.ste100doc(doc)
print(structure)

token_registry = TokenRegistry.Factory.get_instance()
rule_context = RuleContext(vocab, token_registry, TextUtils())
rule_registry = RuleRegistry()
rule_registry.install_rules("biz.dfch.ste100parser.rule_repository")

def _process_tokens(tokens: list[TokenBase]):
    for i, token in enumerate(tokens):
        count = len(tokens)
        print(f"[{i}/{count}] [{type(token).__name__}] Processing token '{token.text}' ...")
        rules = rule_registry.get_rules(token)
        for rule in rules:
            print(f"Processing rule '{rule.rule_id}' [{rule.priority}] [{type(token).__name__}] '{token.text}' ...")
            try:
                test_results = rule.examine(token, rule_context)
                assert isinstance(test_results, list), type(test_results)

                for test_result in test_results:
                    print(f"[{test_result.severity}] {test_result.rule_id}: '{test_result.message}'")
            except Exception as ex:
                print(f"'{type(rule).__name__}' failed [{ex}]")
        if isinstance(token, ListToken):
            print(f"[{type(token).__name__}] Processing token children '{len(token.tokens)}' ...")
            _process_tokens(token.tokens)

_process_tokens(list(doc))

# displacy.render(doc, jupyter=True)


#2 [TEXT]: 'Do'.
#1 [WS]: ' '.
#4 [TEXT]: 'this'.
#1 [WS]: ' '.
#4 [TEXT]: 'test'.
#3 [dquote]: '[Token('DQUOTE', '"'), Tree('TEXT', ['test']), Token('DQUOTE', '"')]'.
#1 [WS]: ' '.
#2 [TEXT]: 'at'.
#1 [WS]: ' '.
#4 [TEXT]: 'full'.
#1 [WS]: ' '.
#10 [paragraph]: '[Tree('TEXT', ['Do']), Tree('WS', ['1']), Tree('TEXT', ['this']), Tree('WS', ['1']), Tree('dquote', [Tree('TEXT', ['test'])]), Tree('WS', ['1']), Tree('TEXT', ['at']), Tree('WS', ['1']), Tree('TEXT', ['full']), Tree('WS', ['1'])]'.
#3 [TEXT]: '100'.
#1 [WS]: ' '.
#5 [TEXT]: 'units'.
#5 [paren]: '[Token('PAREN_OPEN', '('), Tree('TEXT', ['100']), Tree('WS', ['1']), Tree('TEXT', ['units']), Token('PAREN_CLOSE', ')')]'.
#1 [WS]: ' '.
#6 [TEXT]: 'speed.'.
#3 [paragraph]: '[Tree('paren', [Tree('TEXT', ['100']), Tree('WS', ['1']), Tree('TEXT', ['units'])]), Tree('WS', ['1']), Tree('TEXT', ['speed.'])]'.
#2 [start]: '[Tree('paragraph', [Tree('TEXT', ['Do']), Tree('WS', ['1']), Tree('TEXT', ['this']), Tree('WS', ['1']), Tree('dquote', 

Processing rule 'R3.6' [0] [Sentence] 'Do this test at full 100 units speed.' ...
Processing rule 'R5.3' [0] [Sentence] 'Do this test at full 100 units speed.' ...
Processing rule 'R7 - /ACCURACY/AVOIDVAGUETERMS' [0] [Sentence] 'Do this test at full 100 units speed.' ...
[Sentence] Processing token children '14' ...
[0/14] [Word] Processing token 'Do' ...
Processing rule 'R1.1' [0] [Word] 'Do' ...
[0/14] 'Do' [Word]
[11] spacy: '['Do', 'this', '"', 'test', '"', 'at', 'full', '(', ')', 'speed', '.']'
[1/14] [Ws] Processing token ' ' ...
[2/14] [Word] Processing token 'this' ...
Processing rule 'R1.1' [0] [Word] 'this' ...
[2/14] 'this' [Word]
[11] spacy: '['Do', 'this', '"', 'test', '"', 'at', 'full', '(', ')', 'speed', '.']'
[3/14] [Ws] Processing token ' ' ...
[4/14] [Quote] Processing token 'test' ...
[Quote] Processing token children '1' ...
[0/1] [Word] Processing token 'test' ...
Processing rule 'R1.1' [0] [Word] 'test' ...
[None/14] 'test' [Word]
[11] spacy: '['Do', 'this', '"', 

Processing rule 'R3.6' [0] [Sentence] '100 units' ...
Processing rule 'R5.3' [0] [Sentence] '100 units' ...
Processing rule 'R7 - /ACCURACY/AVOIDVAGUETERMS' [0] [Sentence] '100 units' ...
[Sentence] Processing token children '3' ...
[0/3] [Number] Processing token '100' ...
Processing rule 'R1.1' [0] [Number] '100' ...
[0/3] '100' [Number]
[2] spacy: '['100', 'units']'
[1/3] [Ws] Processing token ' ' ...
[2/3] [Text] Processing token 'units' ...
Processing rule 'R1.1' [0] [Text] 'units' ...
[2/3] 'units' [Text]
[2] spacy: '['100', 'units']'
[11/14] [Ws] Processing token ' ' ...
[12/14] [Word] Processing token 'speed' ...
Processing rule 'R1.1' [0] [Word] 'speed' ...
[12/14] 'speed' [Word]
[11] spacy: '['Do', 'this', '"', 'test', '"', 'at', 'full', '(', ')', 'speed', '.']'
[13/14] [Punct] Processing token '.' ...
Processing rule 'R8.1' [85] [Punct] '.' ...


: 